# BirdCLEF+ 2025 - Pretrained models: Perch 2.0 and Bird-MAE-Base embeddings of unlabelled soundscapes (GPU)

This notebook does frozen feature extraction only: no labels, no heads, no training. The files are the 1000 `train_soundscapes` listed in `noise_index.csv` from the baseline's make-mels notebook (seed 42 sample). `test_soundscapes` is never read.

**Windows:** as in the inference notebooks, non-overlapping 5 s windows over the whole file, `n_win = max(1, ceil(len / WIN))`, end padded with zeros. A 1-minute file gives 12, and the real count is recorded. `win` is 0-based, so a window ends at `(win + 1) * 5` seconds.

**Perch 2.0:** the same ONNX file as A2, the same per-window preprocessing, output `embedding`, 1536-d.

**Bird-MAE-Base:** the same files and pinned `transformers` as A3, one feature-extractor call per window, the same mean-pooled 768-d output and the same autocast setting as the A3 embed notebook. These are raw features, without the head mean/std.

**Output:** `perch_soundscape_embeddings.npy`, `birdmae_soundscape_embeddings.npy`, `soundscape_windows.csv` (same row order for both arrays), `soundscape_embed_meta.json`.

In [ ]:
import os, time
import torch

T_NOTEBOOK = time.time()
SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"     # local CPU test only, never set on Kaggle
print("torch.cuda.is_available():", torch.cuda.is_available())
if torch.cuda.is_available():
    print("torch.cuda.get_device_name(0):", torch.cuda.get_device_name(0))
elif SMOKE_TEST:
    print("SMOKE_TEST=1: no GPU, running the local CPU test")
else:
    raise RuntimeError("No GPU found. Enable the T4 accelerator; this notebook never falls back to CPU.")

In [ ]:
import subprocess, sys
from pathlib import Path

OUT = Path(os.environ.get("M3_OUT_DIR", "/kaggle/working"))
ORT_VERSION = "1.31.1"
# Same versions as the A3 embed notebook, in a separate folder so the image's own packages are left alone.
PINS = {"transformers": "4.38.0", "tokenizers": "0.15.2", "huggingface_hub": "0.21.4"}
PYDEPS = Path(os.environ.get("M3_PYDEPS", "/tmp/bmae_pydeps"))

if not SMOKE_TEST:
    # torch is already imported above, which makes its CUDA libraries available to onnxruntime.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "onnxruntime"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"onnxruntime-gpu=={ORT_VERSION}"], check=True)
    print("torch CUDA:", torch.version.cuda, "| cuDNN:", torch.backends.cudnn.version())
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--only-binary=:all:", "--target", str(PYDEPS),
                    *[f"{k}=={v}" for k, v in PINS.items()]], check=True)
sys.path.insert(0, str(PYDEPS))

In [ ]:
import gc, glob, hashlib, json, tempfile
import numpy as np
import pandas as pd
import soundfile as sf
from scipy.signal import resample_poly
import onnxruntime as ort
import transformers
from transformers import AutoFeatureExtractor, AutoModel

print("transformers", transformers.__version__, "from", transformers.__file__)
assert Path(transformers.__file__).is_relative_to(PYDEPS), "pinned transformers is not the one imported"
if not SMOKE_TEST and transformers.__version__ != PINS["transformers"]:
    raise RuntimeError(f"Expected transformers {PINS['transformers']}, got {transformers.__version__}")

# On Kaggle every check is on. M3_STRICT only re-enables named checks inside a local smoke test.
STRICT = set(filter(None, os.environ.get("M3_STRICT", "").split(",")))
def strict(name):
    return not SMOKE_TEST or name in STRICT

SR, WIN_SEC = 32_000, 5
WIN = SR * WIN_SEC          # 160000 samples, the fixed input of both models
PERCH_DIM, BIRDMAE_DIM = 1536, 768
PERCH_SHA256 = "bf0c8467a924cb074663970ca4a0ab1e143602121930209657d0dff5d5cefa1f"
BIRDMAE_SHA256 = "402103ecc81787ca6317ad78377fb38a912b3548f23a18f424d7a0dc31f754c5"   # model.safetensors
MODEL_FILES = ["model.safetensors", "config.json", "preprocessor_config.json",
               "configuration_bird_mae.py", "feature_extractor.py", "modeling_bird_mae.py"]

NW = int(os.environ.get("M3_WORKERS", "0" if SMOKE_TEST else "4"))   # audio-loading workers
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PERCH_BATCH = 8 if SMOKE_TEST else 64
BIRDMAE_BATCH = 8 if SMOKE_TEST else 128
LOADER_FILES = 8            # 96 windows per loader batch, the most the A3 embed loader ever held (16 clips x 6)
MAX_FAILED = 5
EXPECT_FILES = 1000
MIN_WINDOWS = 11000 if strict("n") else 1
BUDGET_S = float(os.environ.get("M3_BUDGET_S", "3600"))        # whole notebook, 1.0 GPU hour
GUARD_AFTER = int(os.environ.get("M3_GUARD_WINDOWS", "500"))
BIRDMAE_WINDOWS_PER_S = 153.0     # logged by the A3 embed run; only used until Bird-MAE has its own rate
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (smoke test)"

def find_dir(marker, env, fallback):
    if os.environ.get(env):
        return Path(os.environ[env])
    hits = glob.glob(f"/kaggle/input/**/{marker}", recursive=True)
    return Path(hits[0]).parent if hits else Path(fallback)

DATA = find_dir("birdclef-2025/sample_submission.csv", "M3_DATA_DIR", "../../data")
MELS = find_dir("noise_index.csv", "M3_MELS_DIR", "../../outputs/jamie_make_mels")
ONNX_PATH = Path(os.environ["M3_PERCH_ONNX"]) if os.environ.get("M3_PERCH_ONNX") \
    else find_dir("perch_v2.onnx", "M3_ONNX_DIR", ".") / "perch_v2.onnx"
MODEL_DIR = find_dir("modeling_bird_mae.py", "M3_BIRDMAE_DIR", ".")
assert ONNX_PATH.exists(), f"perch_v2.onnx not found: attach strippedbanana/sc4000-birdclef-m3-perch-onnx ({ONNX_PATH})"
assert (MODEL_DIR / "model.safetensors").exists(), \
    f"Bird-MAE not found: attach strippedbanana/sc4000-birdclef-m3-birdmae-base ({MODEL_DIR})"
print("Data:", DATA, "| index:", MELS, "| onnx:", ONNX_PATH, "| Bird-MAE:", MODEL_DIR)
print("device:", DEVICE, "| ORT", ort.__version__, ort.get_available_providers())

## File list and model checksums (before any GPU work)

In [ ]:
noise_index = pd.read_csv(MELS / "noise_index.csv")
if "filename" not in noise_index.columns:
    raise RuntimeError(f"noise_index.csv has no filename column: {list(noise_index.columns)}")
names = noise_index["filename"].astype(str).tolist()
dupes = noise_index["filename"].astype(str)[noise_index["filename"].astype(str).duplicated()].tolist()
if dupes:
    raise RuntimeError(f"{len(dupes)} duplicated names in noise_index.csv, first: {dupes[:5]}")
not_bare = [n for n in names if Path(n).name != n or not n.endswith(".ogg")]
if not_bare:
    raise RuntimeError(f"{len(not_bare)} names are not bare .ogg file names, first: {not_bare[:5]}")
if strict("files") and len(names) != EXPECT_FILES:
    raise RuntimeError(f"Expected {EXPECT_FILES} files in noise_index.csv, found {len(names)}")
files = [DATA / "train_soundscapes" / n for n in names]
missing = [n for n, f in zip(names, files) if not f.exists()]
if missing:
    raise RuntimeError(f"{len(missing)} files missing from train_soundscapes, first: {missing[:5]}")
print(len(files), "soundscapes from noise_index.csv")

def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 24), b""):
            h.update(block)
    return h.hexdigest()

t0 = time.time()
perch_sha = sha256_of(ONNX_PATH)
birdmae_file_shas = {n: sha256_of(MODEL_DIR / n) for n in MODEL_FILES}
if strict("sha") and perch_sha != PERCH_SHA256:
    raise RuntimeError(f"perch_v2.onnx checksum mismatch: {perch_sha}")
if strict("sha") and birdmae_file_shas["model.safetensors"] != BIRDMAE_SHA256:
    raise RuntimeError(f"model.safetensors checksum mismatch: {birdmae_file_shas['model.safetensors']}")
print(f"checksums done in {time.time() - t0:.0f}s")

## Shared audio code and the timing guard

In [ ]:
class EmptyAudioError(Exception):
    pass

# Only these count as a decode failure of the file. Anything else is a bug and stops the notebook.
# soundfile >= 0.11 raises SoundFileError subclasses; older versions raise RuntimeError.
DECODE_ERRORS = (getattr(sf, "SoundFileError", RuntimeError), EmptyAudioError)

def to_32k_mono(y, sr):
    """Same as the A2 and A3 notebooks."""
    if y.ndim > 1:
        y = y.mean(axis=1)
    if sr != SR:
        y = resample_poly(y, SR, sr)     # polyphase, as perch-hoplite does
    return y.astype(np.float32)

def preprocess_windows(x):
    """Perch preprocessing, per window: remove the mean, scale the peak to 0.25. Same as the A2 notebooks."""
    x = x - x.mean(axis=1, keepdims=True)
    peak = np.abs(x).max(axis=1, keepdims=True)
    return (x * (0.25 / np.maximum(peak, 1e-8))).astype(np.float32)

def read_windows(path):
    """Whole file, windowed as in the inference notebooks: n_win = ceil(len / WIN), end padded with zeros."""
    y, sr = sf.read(str(path), dtype="float32")
    y = to_32k_mono(y, sr)
    if len(y) == 0:
        raise EmptyAudioError(f"no samples in {Path(path).name}")
    if not np.isfinite(y).all():
        raise ValueError(f"non-finite samples in {Path(path).name}")
    n_win = max(1, int(np.ceil(len(y) / WIN)))
    y = np.pad(y, (0, n_win * WIN - len(y)))
    return np.stack([y[i * WIN:(i + 1) * WIN] for i in range(n_win)])

def add_failure(failed, name, err):
    failed.append((name, err))
    print("decode failure:", name, err, flush=True)
    if len(failed) > MAX_FAILED:
        raise RuntimeError(f"{len(failed)} files failed to decode (limit {MAX_FAILED}), first: {failed[:3]}")

def make_guard(model, later_rate):
    """Once GUARD_AFTER windows have gone through this model, project the whole notebook and stop if it is over
    budget. The rate starts after the first GPU batch, so one-off graph compilation does not inflate it."""
    state = dict(mark=None, checked=False)
    def check(n_done, windows_seen, files_ok):
        if state["checked"]:
            return
        now = time.time()
        if state["mark"] is None:
            state["mark"] = (n_done, now)
            return
        n0, t0 = state["mark"]
        if n_done - n0 < GUARD_AFTER:
            return
        state["checked"] = True
        rate = (n_done - n0) / max(now - t0, 1e-6)
        est_total = windows_seen / max(files_ok, 1) * len(files)
        elapsed = now - T_NOTEBOOK
        projected = elapsed + max(est_total - n_done, 0) / rate + (est_total / later_rate if later_rate else 0.0)
        msg = (f"{model}: {rate:.1f} windows/s after {n_done} windows, about {est_total:.0f} windows in total, "
               f"{elapsed:.0f}s elapsed -> projected notebook time {projected:.0f}s = {projected / 3600:.2f} h "
               f"(limit {BUDGET_S:.0f}s)")
        print("TIMING GUARD |", msg, flush=True)
        if projected > BUDGET_S:
            raise RuntimeError(f"Projected GPU time is over the limit. {msg}")
    return check

## Perch 2.0 on the GPU

In [ ]:
GPU_PROVIDERS = [("CUDAExecutionProvider", {"device_id": 0}), "CPUExecutionProvider"]
PROVIDERS = GPU_PROVIDERS if strict("cuda") else ["CPUExecutionProvider"]

def check_gpu_placement(onnx_path, batch):
    """get_providers() can list CUDA while the graph really runs on CPU, so time each node."""
    so = ort.SessionOptions(); so.enable_profiling = True
    so.profile_file_prefix = str(Path(tempfile.gettempdir()) / "ort_probe")
    s = ort.InferenceSession(str(onnx_path), so, providers=PROVIDERS)
    s.run(["embedding"], {"inputs": batch}); s.run(["embedding"], {"inputs": batch})
    events = json.load(open(s.end_profiling()))
    by = {}
    for e in events:
        if e.get("cat") == "Node" and str(e.get("name", "")).endswith("_kernel_time"):
            p = e.get("args", {}).get("provider", "?")
            by[p] = by.get(p, 0) + e.get("dur", 0)
    total = sum(by.values())
    print("Node kernel time by provider (us):", by)
    if not strict("cuda"):
        return
    share = by.get("CUDAExecutionProvider", 0) / total if total else 0.0
    print(f"CUDA share of node time: {share:.1%}")
    if share < 0.5:
        raise RuntimeError(f"Only {share:.1%} of node time is on CUDA: refusing to run on CPU")

class PerchFiles(torch.utils.data.Dataset):
    def __len__(self):
        return len(files)
    def __getitem__(self, j):
        try:
            return j, preprocess_windows(read_windows(files[j])), ""
        except DECODE_ERRORS as e:         # recorded and limited below
            return j, np.zeros((0, WIN), np.float32), repr(e)

def keep_items(items):
    return items

t_perch_start = time.time()
check_gpu_placement(ONNX_PATH, np.zeros((PERCH_BATCH, WIN), np.float32))
print(f"placement probe {time.time() - t_perch_start:.0f}s")
sess = ort.InferenceSession(str(ONNX_PATH), providers=PROVIDERS)
print("Active providers:", sess.get_providers())
if strict("cuda") and sess.get_providers()[0] != "CUDAExecutionProvider":
    raise RuntimeError(f"CUDAExecutionProvider is not active: {sess.get_providers()}")
assert sess.get_inputs()[0].name == "inputs", sess.get_inputs()[0].name
assert "embedding" in [o.name for o in sess.get_outputs()]

loader = torch.utils.data.DataLoader(
    PerchFiles(), batch_size=LOADER_FILES, shuffle=False, num_workers=NW, collate_fn=keep_items,
    prefetch_factor=4 if NW else None)     # numpy goes straight to ORT, so pin_memory has no effect
perch_guard = make_guard("Perch", BIRDMAE_WINDOWS_PER_S)
embs, perch_j, perch_w, perch_failed = [], [], [], []
pending, n_pending, n_done, n_seen, files_ok = [], 0, 0, 0, 0
t_model = 0.0

def run_batches(x):
    global n_done, t_model
    for i in range(0, len(x), PERCH_BATCH):
        t = time.time()
        b = x[i:i + PERCH_BATCH]
        embs.append(sess.run(["embedding"], {"inputs": b})[0])
        t_model += time.time() - t
        n_done += len(b)
        perch_guard(n_done, n_seen, files_ok)

for step, items in enumerate(loader):
    for j, w, err in items:
        if err:
            add_failure(perch_failed, names[j], err); continue
        perch_j.extend([j] * len(w)); perch_w.extend(range(len(w)))
        pending.append(w); n_pending += len(w); n_seen += len(w); files_ok += 1
    if n_pending >= PERCH_BATCH:
        x = np.concatenate(pending)
        n = len(x) // PERCH_BATCH * PERCH_BATCH
        run_batches(x[:n])
        pending = [x[n:]] if n < len(x) else []; n_pending = len(x) - n
    if step % 25 == 0:
        done = min((step + 1) * LOADER_FILES, len(files)); el = time.time() - t_perch_start
        print(f"Perch {done}/{len(files)} files | {n_done} windows | {el:.0f}s | "
              f"ETA {el / done * (len(files) - done) / 60:.1f} min | model {t_model:.0f}s", flush=True)
if pending:
    run_batches(np.concatenate(pending))

P = np.concatenate(embs).astype(np.float32)
assert P.shape == (len(perch_j), PERCH_DIM), P.shape
perch_wall, perch_model_s = time.time() - t_perch_start, t_model
print(f"Perch done: {P.shape} | failed decodes {len(perch_failed)} | wall {perch_wall:.0f}s | model {perch_model_s:.0f}s")
del sess, embs, loader
gc.collect()

## Bird-MAE-Base on the GPU (raw 768-d features)

In [ ]:
def windows_to_specs(w):
    """One feature-extractor call per window, never a batched call: the repo pads the ~14 missing frames
    with the minimum of whatever batch it is given, so a batched call would make a window's input depend
    on its neighbours and differ between training and inference."""
    return torch.cat([fe(x) for x in w]).float()      # [n, 1, 512, 128]

class BirdMaeFiles(torch.utils.data.Dataset):
    def __len__(self):
        return len(files)
    def __getitem__(self, j):
        try:
            return j, windows_to_specs(read_windows(files[j])), ""
        except DECODE_ERRORS as e:         # recorded and limited below
            return j, torch.zeros((0, 1, 512, 128)), repr(e)

def collate(items):
    specs = torch.cat([it[1] for it in items])
    clips = torch.tensor([it[0] for it in items for _ in range(len(it[1]))], dtype=torch.long)
    wins = torch.tensor([k for it in items for k in range(len(it[1]))], dtype=torch.long)
    return specs, clips, wins, [(it[0], it[2]) for it in items if it[2]]

t_bm_start = time.time()
fe = AutoFeatureExtractor.from_pretrained(str(MODEL_DIR), trust_remote_code=True)
bird_mae, info = AutoModel.from_pretrained(str(MODEL_DIR), trust_remote_code=True, output_loading_info=True)
if info["missing_keys"] or info["unexpected_keys"]:
    raise RuntimeError(f"Weights did not load cleanly: {info['missing_keys'][:5]} {info['unexpected_keys'][:5]}")
assert bird_mae.config.global_pool == "mean", bird_mae.config.global_pool
bird_mae.eval().requires_grad_(False).to(DEVICE)     # frozen: never trained here
use_amp = DEVICE == "cuda"                           # the A3 embed notebook ran under autocast, so this matches it

# Fails in the first minute if the audio path, feature extractor or model do not fit together.
bad_names = {n for n, _ in perch_failed}
probe_file = next(f for n, f in zip(names, files) if n not in bad_names)
probe = windows_to_specs(read_windows(probe_file))
assert probe.shape[1:] == (1, 512, 128), probe.shape
with torch.inference_mode(), torch.autocast(DEVICE, enabled=use_amp):
    out = bird_mae(probe.to(DEVICE))[0]
assert out.shape == (len(probe), BIRDMAE_DIM), out.shape
print("probe OK:", probe.shape, "->", out.shape, "| autocast:", use_amp)

loader = torch.utils.data.DataLoader(
    BirdMaeFiles(), batch_size=LOADER_FILES, shuffle=False, num_workers=NW, collate_fn=collate,
    pin_memory=DEVICE == "cuda", prefetch_factor=4 if NW else None)
bm_guard = make_guard("Bird-MAE", None)
embs, bm_j, bm_w, bm_failed = [], [], [], []
n_done, n_seen, files_ok = 0, 0, 0
t_model = 0.0
for step, (specs, clips, wins, errs) in enumerate(loader):
    for j, e in errs:
        add_failure(bm_failed, names[j], e)
    bm_j.extend(clips.tolist()); bm_w.extend(wins.tolist())
    n_seen += len(specs); files_ok += len(set(clips.tolist()))
    for i in range(0, len(specs), BIRDMAE_BATCH):
        t = time.time()
        x = specs[i:i + BIRDMAE_BATCH].to(DEVICE, non_blocking=True)
        with torch.inference_mode(), torch.autocast(DEVICE, enabled=use_amp):
            e = bird_mae(x)[0]
        embs.append(e.float().cpu().numpy())
        t_model += time.time() - t
        n_done += len(x)
        bm_guard(n_done, n_seen, files_ok)
    if step % 25 == 0:
        done = min((step + 1) * LOADER_FILES, len(files)); el = time.time() - t_bm_start
        print(f"Bird-MAE {done}/{len(files)} files | {n_done} windows | {el:.0f}s | "
              f"ETA {el / done * (len(files) - done) / 60:.1f} min | model {t_model:.0f}s", flush=True)

B = np.concatenate(embs).astype(np.float32)
assert B.shape == (len(bm_j), BIRDMAE_DIM), B.shape
bm_wall, bm_model_s = time.time() - t_bm_start, t_model
print(f"Bird-MAE done: {B.shape} | failed decodes {len(bm_failed)} | wall {bm_wall:.0f}s | model {bm_model_s:.0f}s")
del bird_mae, embs, loader
if DEVICE == "cuda":
    torch.cuda.empty_cache()

## Checks, then atomic writes

In [ ]:
if list(zip(perch_j, perch_w)) != list(zip(bm_j, bm_w)):
    raise RuntimeError("Perch and Bird-MAE windows differ: the two arrays would not be aligned")
if {n for n, _ in perch_failed} != {n for n, _ in bm_failed}:
    raise RuntimeError(f"Decode failures differ between models: {perch_failed} vs {bm_failed}")

windows = pd.DataFrame({"filename": [names[j] for j in perch_j], "win": perch_w})
N = len(windows)
assert P.shape == (N, PERCH_DIM) and B.shape == (N, BIRDMAE_DIM), (P.shape, B.shape, N)
for label, A in (("Perch", P), ("Bird-MAE", B)):
    if not np.isfinite(A).all():
        raise RuntimeError(f"{label} embeddings contain NaN or Inf")
if N <= MIN_WINDOWS:
    raise RuntimeError(f"Only {N} windows, expected more than {MIN_WINDOWS}")
# Every embedded file must have windows 0..n-1 with no gap and no repeat.
for name, g in windows.groupby("filename", sort=False):
    assert g["win"].tolist() == list(range(len(g))), f"windows of {name} are not 0..n-1"

def variance_summary(A):
    v = A.var(axis=0)
    return dict(min=float(v.min()), median=float(np.median(v)), max=float(v.max()), trace=float(v.sum()),
                n_dims_below_1e8=int((v < 1e-8).sum()))

per_file = windows.groupby("filename", sort=False).size()
wall_total = time.time() - T_NOTEBOOK
meta = dict(
    perch_sha256=perch_sha, birdmae_files_sha256=birdmae_file_shas, sha_checked=strict("sha"),
    transformers_pin=PINS, onnxruntime_version=ort.__version__, device_name=GPU_NAME, autocast=use_amp,
    files_listed=len(names), files_embedded=int(len(per_file)), windows=N,
    windows_per_file_counts={str(int(k)): int(v) for k, v in per_file.value_counts().items()},
    failed_files=[dict(filename=n, error=e) for n, e in perch_failed],
    wall_s=dict(perch=perch_wall, birdmae=bm_wall, notebook=wall_total),
    model_s=dict(perch=perch_model_s, birdmae=bm_model_s),
    variance=dict(perch=variance_summary(P), birdmae=variance_summary(B)))

def save_npy(name, A):
    tmp = OUT / (name[:-4] + ".tmp.npy")
    np.save(tmp, A)
    os.replace(tmp, OUT / name)

save_npy("perch_soundscape_embeddings.npy", P)
save_npy("birdmae_soundscape_embeddings.npy", B)
tmp = OUT / "soundscape_windows.tmp.csv"
windows.to_csv(tmp, index=False)
os.replace(tmp, OUT / "soundscape_windows.csv")
tmp = OUT / "soundscape_embed_meta.tmp.json"          # written last: its presence means the run is complete
with open(tmp, "w") as f:
    json.dump(meta, f, indent=1)
os.replace(tmp, OUT / "soundscape_embed_meta.json")

print("Windows per file:", meta["windows_per_file_counts"], "| failed files:", len(perch_failed))
print(f"TIMING | {N} windows, {len(per_file)} files | Perch wall {perch_wall:.0f}s, {N / perch_wall:.1f} windows/s | "
      f"Bird-MAE wall {bm_wall:.0f}s, {N / bm_wall:.1f} windows/s | notebook {wall_total:.0f}s = {wall_total / 3600:.2f} h")